# 🧬 Chromosome Classification — MobileNetV2
## Notebook 4 of 5
**Estimated time on Kaggle P100:** ~~30 min
**After this notebook:** Download `mobilenetv2_best.keras` and `mobilenetv2_test_probs.npy`
from Output tab → upload to a new Kaggle Dataset → use in Notebook 6 (Ensemble)

## Cell 1 — Setup + Imports

In [2]:
# Cell 0A — Download dataset into Colab
import os
from google.colab import files

print("📁 Upload your kaggle.json file...")
uploaded = files.upload()

# Get whatever filename was uploaded (handles kaggle(1).json etc)
uploaded_name = list(uploaded.keys())[0]
print(f"Uploaded as: {uploaded_name}")

# Place in correct location regardless of filename
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded[uploaded_name])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ kaggle.json placed correctly")

# Install kaggle CLI
!pip install -q kaggle

# Check if dataset already downloaded to save time
if os.path.exists('/content/chromo_data/Data/single_chromosomes_object/JEPG'):
    print("✅ Dataset already exists — skipping download")
else:
    print("📥 Downloading dataset (~2-3 minutes)...")
    !kaggle datasets download \
        -d aliabedimadiseh/chromosome-image-dataset-karyotype \
        --path /content/chromo_data \
        --unzip -q
    print("✅ Download complete")

# Verify
img_dir = '/content/chromo_data/Data/single_chromosomes_object/JEPG'
if os.path.exists(img_dir):
    count = len(os.listdir(img_dir))
    print(f"✅ Images found: {count} (should be ~2986)")
else:
    print("❌ Image folder not found — check path")
    for root, dirs, files in os.walk('/content/chromo_data'):
        level = root.replace('/content/chromo_data','').count(os.sep)
        if level > 3: continue
        print('  '*level + os.path.basename(root) + '/')

📁 Upload your kaggle.json file...


Saving kaggle.json to kaggle.json
Uploaded as: kaggle.json
✅ kaggle.json placed correctly
📥 Downloading dataset (~2-3 minutes)...
Dataset URL: https://www.kaggle.com/datasets/aliabedimadiseh/chromosome-image-dataset-karyotype
License(s): CC-BY-SA-4.0
✅ Download complete
✅ Images found: 2000 (should be ~2986)


In [3]:
# ── NEW CELL 0B: Verify GPU ──────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f'✅ GPU active: {gpus[0].name}')
    print(f'   Make sure you selected GPU in Runtime → Change runtime type')
else:
    print('⚠️  NO GPU DETECTED')
    print('   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   Then run this cell again')

tf.keras.mixed_precision.set_global_policy('float32')
print('✅ Precision: float32')

✅ GPU active: /physical_device:GPU:0
   Make sure you selected GPU in Runtime → Change runtime type
✅ Precision: float32


In [4]:
import tensorflow as tf, os, re, random, warnings, time, gc, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as mpatches
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              accuracy_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    [tf.config.experimental.set_memory_growth(g, True) for g in gpus]
    print(f'✅ GPU: {gpus[0].name}')
else:
    print('💻 CPU mode')
tf.keras.mixed_precision.set_global_policy('float32')
print('✅ Setup complete')

from tensorflow.keras.applications import MobileNetV2

✅ GPU: /physical_device:GPU:0
✅ Setup complete


## Cell 2 — Config + Load labels + Split

In [5]:
# Cell 2 — Config + Load Labels + Split (Normal vs Abnormal karyogram classification)
import os, numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DATA_ROOT = '/content/chromo_data/Data'
IMG_DIR   = os.path.join(DATA_ROOT, '24_chromosomes_object', 'JEPG')
assert os.path.exists(IMG_DIR), f"Folder missing: {IMG_DIR} — check download in Cell 0A"

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 2
OUTPUT_DIR  = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load labels from CSV
norm_df = pd.read_csv(os.path.join(DATA_ROOT, 'normal.csv'), header=0)
nabn_df = pd.read_csv(os.path.join(DATA_ROOT, 'number_abnormalities.csv'), header=0)
sabn_df = pd.read_csv(os.path.join(DATA_ROOT, 'structural_abnormalities.csv'), header=0)

normal_files = set(norm_df.iloc[:, 0].astype(str).str.strip())
nabn_files   = set(nabn_df.iloc[:, 0].astype(str).str.strip())
sabn_files   = set(sabn_df.iloc[:, 0].astype(str).str.strip())
abn_files    = nabn_files | sabn_files

print(f"Normal images   : {len(normal_files)}")
print(f"Abnormal images : {len(abn_files)}")

# Build records: label 0=normal, 1=abnormal
records = []
for fname in sorted(os.listdir(IMG_DIR)):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    fp = os.path.join(IMG_DIR, fname)
    if fname in abn_files:
        records.append((fp, 1))
    elif fname in normal_files:
        records.append((fp, 0))
    # skip unlabeled files

print(f"\nTotal labeled : {len(records)}")
dist = Counter(r[1] for r in records)
print(f"Normal  (0)   : {dist[0]}")
print(f"Abnormal (1)  : {dist[1]}")

fps    = [r[0] for r in records]
labels = [r[1] for r in records]

fps_train, fps_val, lbl_train, lbl_val = train_test_split(
    fps, labels, test_size=0.20, random_state=42, stratify=labels)

train_records = list(zip(fps_train, lbl_train))
val_records   = list(zip(fps_val,   lbl_val))
test_records  = val_records
lbl_test      = lbl_val

cls_wts = compute_class_weight('balanced', classes=np.array([0, 1]), y=lbl_train)
class_weight_dict = {0: cls_wts[0], 1: cls_wts[1]}
steps = len(train_records) // BATCH_SIZE

print(f"\nTrain : {len(train_records)}")
print(f"Val   : {len(val_records)}")
print(f"Steps/epoch: {steps}")
print(f"Class weights: Normal={cls_wts[0]:.2f}, Abnormal={cls_wts[1]:.2f}")

Normal images   : 4893
Abnormal images : 109

Total labeled : 4994
Normal  (0)   : 4891
Abnormal (1)  : 103

Train : 3995
Val   : 999
Steps/epoch: 124
Class weights: Normal=0.51, Abnormal=24.36


## Cell 3 — tf.data Pipeline

In [6]:
# Cell 3 — tf.data Pipeline (EfficientNetB2 version, with oversampling)
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
AUTOTUNE = tf.data.AUTOTUNE

IMG_SIZE = 260   # EfficientNetB2's native input size (not 224 like the others)

def preprocess_tf(filepath, label):
    raw = tf.io.read_file(filepath)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = efficientnet_preprocess(img)   # EfficientNet expects raw 0-255 input, normalizes internally
    return img, tf.one_hot(label, NUM_CLASSES)

def augment_fn(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    return image, label

norm_train_fps = [fp for fp, lbl in train_records if lbl == 0]
abn_train_fps  = [fp for fp, lbl in train_records if lbl == 1]
print(f"Train — Normal: {len(norm_train_fps)}  Abnormal: {len(abn_train_fps)}")

norm_ds = tf.data.Dataset.from_tensor_slices((norm_train_fps, [0]*len(norm_train_fps))).shuffle(len(norm_train_fps), seed=42).repeat()
abn_ds  = tf.data.Dataset.from_tensor_slices((abn_train_fps,  [1]*len(abn_train_fps))).shuffle(len(abn_train_fps), seed=42).repeat()

train_ds = tf.data.Dataset.sample_from_datasets(
    [norm_ds, abn_ds], weights=[0.5, 0.5], seed=42
).map(preprocess_tf, num_parallel_calls=AUTOTUNE) \
 .map(augment_fn, num_parallel_calls=AUTOTUNE) \
 .batch(BATCH_SIZE) \
 .prefetch(AUTOTUNE)

val_ds = (
    tf.data.Dataset.from_tensor_slices(([r[0] for r in val_records], [r[1] for r in val_records]))
    .map(preprocess_tf, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = val_ds
steps = len(train_records) // BATCH_SIZE
print(f"✅ Pipeline ready | Steps/epoch: {steps} | IMG_SIZE={IMG_SIZE}")

Train — Normal: 3913  Abnormal: 82
✅ Pipeline ready | Steps/epoch: 124 | IMG_SIZE=260


## Cell 4 — Model

In [7]:
# Cell 4 — EfficientNetB2 binary classifier
from tensorflow.keras.applications import EfficientNetB2
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

def build_binary_model():
    inp  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = EfficientNetB2(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False

    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_initializer='he_uniform',
                     kernel_regularizer=regularizers.l2(2e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(2, activation='softmax', dtype='float32', name='output')(x)
    return Model(inp, out), base

model, backbone = build_binary_model()
total = sum(v.numpy().size for v in model.weights) / 1e6
print(f"✅ EfficientNetB2 model ready | Total params: {total:.1f}M")

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ EfficientNetB2 model ready | Total params: 8.1M


## Cell 5 — Training (Phase 1 +  Phase 2)

In [8]:
# Cell 5 — Two-phase training
ckpt = f'{OUTPUT_DIR}/efficientnetb2_binary_best.keras'
loss_fn = keras.losses.CategoricalFocalCrossentropy(gamma=2.0, alpha=0.25)
metrics = ['accuracy', keras.metrics.AUC(name='auc'),
           keras.metrics.Precision(name='precision', class_id=1),
           keras.metrics.Recall(name='recall', class_id=1)]

backbone.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss_fn, metrics=metrics)
cb1 = [
    keras.callbacks.ModelCheckpoint(ckpt, monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]
print("=== Phase 1: frozen backbone ===")
h1 = model.fit(train_ds, validation_data=val_ds, epochs=20, steps_per_epoch=steps, callbacks=cb1, verbose=1)

backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5, clipnorm=1.0), loss=loss_fn, metrics=metrics)
cb2 = [
    keras.callbacks.ModelCheckpoint(ckpt, monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-8, verbose=1),
]
print("=== Phase 2: fine-tuning ===")
h2 = model.fit(train_ds, validation_data=val_ds, epochs=60, steps_per_epoch=steps, callbacks=cb2, verbose=1)
model.load_weights(ckpt)
print(f"✅ Best val_auc: {max(h1.history['val_auc'] + h2.history['val_auc']):.4f}")

=== Phase 1: frozen backbone ===
Epoch 1/20
124/124 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.6038 - auc: 0.6492 - loss: 0.2633 - precision: 0.6184 - recall: 0.6155
Epoch 1: val_auc improved from None to 0.90032, saving model to /kaggle/working/efficientnetb2_binary_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnetb2_binary_best.keras
124/124 ━━━━━━━━━━━━━━━━━━━━ 105s 474ms/step - accuracy: 0.6792 - auc: 0.7404 - loss: 0.2132 - precision: 0.6869 - recall: 0.6886 - val_accuracy: 0.8188 - val_auc: 0.9003 - val_loss: 0.1245 - val_precision: 0.0402 - val_recall: 0.3333 - learning_rate: 0.0010
Epoch 2/20
124/124 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.7449 - auc: 0.8299 - loss: 0.1559 - precision: 0.7556 - recall: 0.7526
Epoch 2: val_auc improved from 0.90032 to 0.93624, saving model to /kaggle/working/efficientnetb2_binary_best.keras

Epoch 2: finished saving model to /kaggle/working/efficientnetb2_binary_best.keras
124/124 ━━━━━━━━━━━━━━━━━━━━ 19s 

## Cell 6 — Evaluate + Save

In [9]:
# Cell 6 — Evaluate with tuned threshold
from sklearn.metrics import precision_recall_curve, classification_report
import numpy as np

probs = model.predict(test_ds, verbose=1)
true  = np.array(lbl_test)
abn_probs = probs[:, 1]

prec, rec, thr = precision_recall_curve(true, abn_probs)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
best_idx = np.argmax(f1s[:-1])
best_thr = thr[best_idx]
print(f"Best threshold: {best_thr:.3f}")

preds = (abn_probs >= best_thr).astype(int)
print(classification_report(true, preds, target_names=['Normal','Abnormal'], zero_division=0))

np.save(f'{OUTPUT_DIR}/efficientnetb2_test_probs.npy', probs)
np.save(f'{OUTPUT_DIR}/efficientnetb2_lbl_test.npy', true)
print('✅ Saved — efficientnetb2_test_probs.npy + efficientnetb2_lbl_test.npy')

32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 398ms/step
Best threshold: 0.856
              precision    recall  f1-score   support

      Normal       0.98      1.00      0.99       978
    Abnormal       1.00      0.10      0.17        21

    accuracy                           0.98       999
   macro avg       0.99      0.55      0.58       999
weighted avg       0.98      0.98      0.97       999

✅ Saved — efficientnetb2_test_probs.npy + efficientnetb2_lbl_test.npy


Tabular Output

In [10]:
# Cell 7 — Per-Class Normal/Abnormal Summary Table (dataset-derived, test set)
import pandas as pd
import re

norm_df = pd.read_csv(os.path.join(DATA_ROOT, 'normal.csv'), header=0)
nabn_df = pd.read_csv(os.path.join(DATA_ROOT, 'number_abnormalities.csv'), header=0)
sabn_df = pd.read_csv(os.path.join(DATA_ROOT, 'structural_abnormalities.csv'), header=0)

CLASS_NAMES = [str(i) for i in range(1, 23)] + ['X', 'Y']
class_normal_count   = {c: 0 for c in CLASS_NAMES}
class_abnormal_count = {c: 0 for c in CLASS_NAMES}

test_fnames = set(os.path.basename(fp) for fp in [r[0] for r in val_records])

norm_test = norm_df[norm_df.iloc[:, 0].astype(str).isin(test_fnames)]
for _ in range(len(norm_test)):
    for c in CLASS_NAMES[:22]:
        class_normal_count[c] += 2
    class_normal_count['X'] += 2
    class_normal_count['Y'] += 0

def extract_classes(desc):
    if pd.isna(desc):
        return []
    found = []
    for tok in re.split(r'[,/ ]+', str(desc)):
        tok = tok.strip().upper()
        if tok in CLASS_NAMES:
            found.append(tok)
    return found

for df in [nabn_df, sabn_df]:
    df_test = df[df.iloc[:, 0].astype(str).isin(test_fnames)]
    for _, row in df_test.iterrows():
        classes = extract_classes(row.iloc[2] if len(row) > 2 else None)
        if not classes:
            continue
        for c in classes:
            class_abnormal_count[c] += 1
        for c in CLASS_NAMES[:22]:
            if c not in classes:
                class_normal_count[c] += 2

summary = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Normal Count': [class_normal_count[c] for c in CLASS_NAMES],
    'Abnormal Count': [class_abnormal_count[c] for c in CLASS_NAMES],
})
summary['Total'] = summary['Normal Count'] + summary['Abnormal Count']
summary['% Abnormal'] = (summary['Abnormal Count'] / summary['Total'].replace(0, 1) * 100).round(2)

print("=== Per-Chromosome-Class Normal/Abnormal Summary (Test Set) — EfficientNetB2 run ===")
print("NOTE: derived from dataset CSV records, not per-chromosome model inference\n")
print(summary.to_string(index=False))

summary.to_csv(f'{OUTPUT_DIR}/efficientnetb2_per_class_summary.csv', index=False)
print(f"\n✅ Saved: {OUTPUT_DIR}/efficientnetb2_per_class_summary.csv")

=== Per-Chromosome-Class Normal/Abnormal Summary (Test Set) — EfficientNetB2 run ===
NOTE: derived from dataset CSV records, not per-chromosome model inference

Class  Normal Count  Abnormal Count  Total  % Abnormal
    1          1996               0   1996        0.00
    2          1996               0   1996        0.00
    3          1994               1   1995        0.05
    4          1996               0   1996        0.00
    5          1996               0   1996        0.00
    6          1996               0   1996        0.00
    7          1996               0   1996        0.00
    8          1994               1   1995        0.05
    9          1992               2   1994        0.10
   10          1996               0   1996        0.00
   11          1996               0   1996        0.00
   12          1996               0   1996        0.00
   13          1994               1   1995        0.05
   14          1992               2   1994        0.10
   15         

## ✅ MobileNetV2 Complete
**Download from Output tab:**
- `mobilenetv2_best.keras` — trained model weights
- `mobilenetv2_test_probs.npy` — test set probability outputs (needed for ensemble)
- `lbl_test.npy` — true test labels (needed for ensemble)
- `mobilenetv2_curves.png` — training curves
- `mobilenetv2_confusion.png` — confusion matrix